# 🛡️ AI Security Monitor — run the full dashboard from GitHub

**One notebook. One click.** It clones the project from your GitHub repository, installs everything, (optionally) trains the models, and opens the **full dashboard with all 5 tabs** — *Overview · Network traffic · HTTP requests (NLP) · Model quality · About* — printing a **public link** you can open in a browser.

### How to use (Google Colab)
1. In **Cell 1** below, put your GitHub repo URL into `REPO_URL` (see the GitHub setup steps at the bottom of this notebook).
2. `Runtime → Run all`.
3. The last cell prints a link like `https://…trycloudflare.com` — open it.

> No GitHub yet? Set `USE_UPLOAD = True` in Cell 1 and it will ask you to upload `ai_security_monitor_dashboard.zip` instead — so you are never blocked.

## Cell 1 — get the project (from GitHub, or by upload)

In [ ]:
import os, zipfile

# ====== EDIT THIS ======
REPO_URL   = "https://github.com/YOUR-USERNAME/ai-security-monitor.git"   # <-- your repo
USE_UPLOAD = False   # set True to upload the ZIP instead of cloning from GitHub
# =======================

if USE_UPLOAD:
    from google.colab import files
    print("Upload ai_security_monitor_dashboard.zip ...")
    up = files.upload()
    name = next(k for k in up if k.endswith(".zip"))
    with zipfile.ZipFile(name) as z: z.extractall(".")
else:
    os.system(f"git clone {REPO_URL} repo")
    # the project folder may be the repo root or a subfolder
    import glob
    hits = glob.glob("**/app.py", recursive=True)
    if not hits:
        raise SystemExit("app.py not found — check REPO_URL or set USE_UPLOAD=True")

# find the project folder (the one that contains app.py) and enter it
import glob
app_path = sorted(glob.glob("**/app.py", recursive=True), key=len)[0]
os.chdir(os.path.dirname(app_path) or ".")
print("Project folder:", os.getcwd())
print("Contents:", sorted(os.listdir(".")))

## Cell 2 — install dependencies

In [ ]:
!pip install -q streamlit joblib scikit-learn seaborn matplotlib pandas

## Cell 3 — (optional) train the models

If the repo already contains `models/` and `outputs/`, you can **skip this** — the dashboard will show results immediately. Run it to reproduce training from scratch (~1–2 min); it also generates the network dataset if missing.

In [ ]:
import os
# generate the synthetic network dataset if it is not in the repo
if not os.path.exists("data/network_traffic.csv"):
    os.system("python scripts/generate_network_data.py")
# train both modules (network + NLP), rebuild metrics and event logs
os.system("python train_all.py")
print("Training finished.")

## Cell 4 — launch the dashboard and print the public link

Starts the Streamlit server and opens a public tunnel. Click the printed link to open the full 5-tab dashboard.

> If no link appears within ~40 s, run this cell again (the free tunnel sometimes needs a second try).

In [ ]:
import subprocess, time, re, os

if not os.path.exists("cloudflared"):
    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared")

subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501", "--server.headless", "true"])
time.sleep(8)

proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url, t0 = None, time.time()
for line in proc.stdout:
    m = re.search(r"https://[-\w.]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
    if time.time() - t0 > 45:
        break

print("\n" + "=" * 54)
print("  OPEN THE DASHBOARD:", url)
print("  5 tabs: Overview | Network traffic | HTTP requests |")
print("          Model quality | About")
print("=" * 54)

## How to put the project on GitHub (one-time, ~3 minutes)

**Easiest way — upload in the browser (no git commands):**
1. Go to **github.com → New repository**. Name it e.g. `ai-security-monitor`, keep it **Public**, click *Create repository*.
2. On the new repo page click **“uploading an existing file”**.
3. Unzip `ai_security_monitor_dashboard.zip` on your computer and **drag the contents of the `ai_security_monitor` folder** (app.py, config.py, src/, data/, models/, outputs/, requirements.txt, …) into the upload area. Click **Commit changes**.
4. Copy the repo URL (green **Code** button → HTTPS), e.g. `https://github.com/your-username/ai-security-monitor.git`.
5. Paste it into `REPO_URL` in Cell 1 above and run the notebook.

**Open this launcher straight from GitHub (optional):** in Colab choose *File → Open notebook → GitHub*, paste your repo URL, and pick this `.ipynb` if you commit it too.

Notes for the defense: the link is live only while this Colab cell runs; user text in the dashboard is never executed (no `unsafe_allow_html`); every result is labeled “Model prediction (not confirmed)”.